# Example: Bandit Elasticity Learning

In this example, we apply the **epsilon-greedy multi-armed bandit** to learn the best CES elasticity $\eta$ per sentiment regime. Instead of using the Session 2 heuristic $\eta(\lambda_t) = \eta_{\min} + (\eta_{\max} - \eta_{\min})/(1 + |\lambda_t|)$, we let the bandit discover which $\eta$ value produces the highest utility in bearish, neutral, and bullish conditions.

> __Learning Objectives:__
>
> * __Train the elasticity bandit on a single path:__ Run the epsilon-greedy bandit over a discrete elasticity grid with separate instances per regime bin. Visualize reward convergence, exploration decay, and per-regime arm means.
> * __Backtest bandit-learned elasticity across Monte Carlo paths:__ Compare three CES strategies (bandit-learned elasticity, heuristic elasticity, fixed elasticity) on the same ensemble from Example 1. Evaluate the distributional performance to see which strategy delivers the best median terminal wealth, drawdown, and Sharpe.
> * __Compare learned vs. heuristic elasticity head-to-head:__ Produce a scorecard and paired-excess analysis showing where the bandit outperforms and where it underperforms the heuristic. Read the win rate and median excess to judge whether the learned elasticity is worth the training cost.

Let's see if a learner can beat a heuristic.

___

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

### Constants


In [ ]:
# Eta-bandit configuration
B₀ = 10_000.0
Δt = 1.0 / 252.0
L_short = 21
L_long = 63
L_growth = 10
GAIN = 10.0
offset = L_short + L_long
SCENARIO_SEED = 2026
ETA_GRID = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
BANDIT_ITERS_SINGLE = 500
BANDIT_ALPHA = 0.1
LAMBDA_THRESHOLD = 0.5
TRIGGER_MAX_DRAWDOWN = 0.15
TRIGGER_MAX_TURNOVER = 0.50
ETA_BOUNDS = (0.5, 5.0)
ALLOCATION_EPSILON = 0.1

Load the same Session 1 artifacts and regenerate the same 500-path scenario used in Example 1 (EWLS Engine Replay). The deterministic seed ensures identical paths.

In the code block below, we populate the following top-level bindings:

| Binding | Type | Description |
|---|---|---|
| `my_tickers` | `Vector{String}` | Session 1 ticker universe (same order as `allocation_weights`). |
| `sim_estimates` | [`Vector{MySIMParameterEstimate}`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MySIMParameterEstimate) | per-ticker SIM fits $(\alpha, \beta, \sigma_\varepsilon)$. |
| `sim_params` | `Dict{String,Tuple{Float64,Float64,Float64}}` | allocator-adapter view of `sim_estimates`, keyed by ticker. |
| `market_model` | [`MyMarketSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyMarketSurrogateModel) | 50-state HMM market surrogate fit to SPY. |
| `portfolio` | [`MyPortfolioSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyPortfolioSurrogateModel) | per-ticker marginal HMMs + Student-$t$ copula for path composition. |
| `calib` | `Dict{String,Any}` | full SIM calibration dict (source of truth for α, β, σ_ε, bootstrap SEs). |
| `start_prices` | `Dict{String,Float64}` | per-ticker starting prices for forward scenarios. |
| `N` | `Int` | number of tickers. |
| `scenario` | [`MyBacktestScenario`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyBacktestScenario) | 500-path × $T$-step hybrid forward scenario (seed-locked for reproducibility across Examples 1-2). |

In [ ]:
my_tickers, sim_estimates, sim_params, market_model, portfolio, calib, start_prices, N, scenario = let
    # --- Step 1: Load S1 artifacts ---
    minvar = load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"));
    my_tickers    = minvar["my_tickers"]::Vector{String};
    sim_estimates = minvar["sim_estimates"];
    sim_params = Dict{String,Tuple{Float64,Float64,Float64}}(
        e.ticker => (e.α, e.β, e.σ_ε) for e in sim_estimates
    );

    # --- Step 2: Surrogates and scenario ---
    market_model = MyMarketSurrogateModel();
    portfolio    = MyPortfolioSurrogateModel();
    calib        = load_results(joinpath(_PATH_TO_DATA_S1, "sim-parameter-estimates.jld2"));
    snap        = MyCurrentPrices();
    snap_lookup = Dict(snap["tickers"] .=> snap["prices"]);
    start_prices = Dict(t => snap_lookup[t] for t in my_tickers);

    # --- Step 3: Dimensions ---
    N         = length(my_tickers);
    n_trading_days   = 252;
    T_total          = offset + n_trading_days;

    # --- Step 4: Regenerate scenario (same seed as Example 1) ---
    Random.seed!(SCENARIO_SEED);
    scenario = generate_hybrid_scenario(
        market_model, portfolio, calib, my_tickers;
        n_paths = 500, n_steps = T_total,
        start_prices = start_prices, label = "S3 Eta-Bandit", seed = SCENARIO_SEED);

    println("Loaded $(N) tickers, generated $(scenario.n_paths)-path scenario")
    my_tickers, sim_estimates, sim_params, market_model, portfolio, calib, start_prices, N, scenario
end

___
## Task 1: Train the Elasticity Bandit on a Single Path
In this task, we extract the first path from the scenario, build the rebalancing context, and run the elasticity bandit with 6 arms ($\eta \in \{0.5, 1.0, 1.5, 2.0, 3.0, 5.0\}$) and a regime threshold $\theta = 0.5$. The bandit learns independently in bearish, neutral, and bullish bins, calling [`solve_eta_bandit(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.solve_eta_bandit) on a [`MyEtaBanditModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyEtaBanditModel) instance.

> __What should we see?__
>
> The reward trace should show initial exploration (noisy) converging to a stable level as the bandit discovers the best arm per regime. The per-regime bar chart should reveal whether different regimes prefer different elasticity values.

The code block below returns `bandit_result::`[`MyEtaBanditResult`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyEtaBanditResult) carrying the per-regime arm means, pull counts, reward trace, and best-elasticity map.

In [ ]:
bandit_result = let
    # --- Step 1: Extract first path ---
    p_idx = 1;
    T_total = scenario.n_steps;
    mkt = scenario.market_paths[p_idx, :];
    pmatrix = zeros(T_total, N + 1);
    pmatrix[:, 1] = 1:T_total;
    for k in 1:N
        pmatrix[:, k + 1] = scenario.price_paths[p_idx, :, k];
    end

    # --- Step 2: Compute lambda series ---
    ema_s = compute_ema(mkt; window = L_short);
    ema_l = compute_ema(mkt; window = L_long);
    lambda_series = compute_lambda(ema_s, ema_l; G = GAIN);
    lambda_series[1:offset] .= 0.0;
    gm_raw = compute_market_growth(mkt; Δt = Δt);
    gm_ema = compute_ema(gm_raw; window = L_growth);

    # --- Step 3: Build context ---
    ctx = build(MyRebalancingContextModel, (
        B = B₀, tickers = my_tickers, marketdata = pmatrix,
        marketfactor = gm_ema, sim_parameters = sim_params,
        lambda = 0.0, Δt = Δt, epsilon = ALLOCATION_EPSILON
    ));

    # --- Step 4: Build and run eta-bandit ---
    η_grid = ETA_GRID;
    bandit = build(MyEtaBanditModel, (
        eta_grid = η_grid, n_iterations = BANDIT_ITERS_SINGLE, alpha = BANDIT_ALPHA, lambda_threshold = LAMBDA_THRESHOLD
    ));

    time_indices = collect((offset + 1):T_total);
    bandit_result = solve_eta_bandit(bandit, ctx, lambda_series, time_indices);

    # --- Step 5: Report best eta per regime ---
    println("Eta-Bandit Results (Path $(p_idx)):")
    for regime in [:bearish, :neutral, :bullish]
        η_best = bandit_result.best_eta_per_regime[regime];
        counts = bandit_result.arm_counts_per_regime[regime];
        means = bandit_result.arm_means_per_regime[regime];
        println("  $(regime): best η = $(η_best), pulls = $(sum(counts)), arm means = $(round.(means, digits=3))")
    end

    # --- Panel 1: Reward trace ---
    p1 = plot(bandit_result.reward_history, label="Reward", alpha=0.3, color=:gray50,
        ylabel="CES Utility", title="Eta-Bandit: Reward Trace", size=(800, 250))

    # --- Panel 2: Exploration decay ---
    p2 = plot(bandit_result.exploration_history, label="ε_t", color=:coral, linewidth=2,
        ylabel="ε", title="Exploration Decay", xlabel="Iteration", size=(800, 200))

    display(plot(p1, p2, layout=grid(2,1, heights=[0.55, 0.45]), size=(800, 450)))

    # --- Panel 3: Per-regime arm means ---
    p3 = plot(layout=(1,3), size=(900, 300), title=["Bearish" "Neutral" "Bullish"])
    for (j, regime) in enumerate([:bearish, :neutral, :bullish])
        bar!(p3[j], string.(η_grid), bandit_result.arm_means_per_regime[regime],
            label="", color=:steelblue, xlabel="η", ylabel=j==1 ? "Mean Reward" : "")
    end
    display(p3)
    bandit_result
end

___
## Task 2: Backtest Bandit-Learned Elasticity Across All Paths
In this task, we take the elasticity map learned on path 1 and apply it across all 500 paths via [the `backtest_eta_bandit(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_eta_bandit). We also run the heuristic $\eta(\lambda)$ engine and a fixed $\eta = 2.0$ engine as baselines.

> __What should we see?__
>
> The bandit-learned elasticity should match or beat the heuristic on median metrics. The fixed elasticity baseline should underperform both adaptive approaches, confirming that regime-awareness matters.

The code block below returns `bandit_bt::`[`MyBacktestResult`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyBacktestResult), carrying per-path terminal wealth, drawdown, and Sharpe arrays for the bandit-learned-elasticity engine run across the 500-path ensemble.

In [ ]:
bandit_bt = let
    # --- Step 1: Run bandit-learned eta engine ---
    eta_map = bandit_result.best_eta_per_regime;
    bandit_bt = backtest_eta_bandit(scenario, my_tickers, sim_params, eta_map;
        B₀ = B₀, offset = offset, L_short = L_short, L_long = L_long,
        GAIN = GAIN, L_growth = L_growth, lambda_threshold = LAMBDA_THRESHOLD);

    # --- Step 2: Run heuristic eta engine (adaptive_eta=true) ---
    n_trading = scenario.n_steps - offset;
    heur_final_wealth = zeros(scenario.n_paths);
    heur_max_dd = zeros(scenario.n_paths);
    heur_sharpe = zeros(scenario.n_paths);

    for p in 1:scenario.n_paths
        mkt = scenario.market_paths[p, :];
        ema_s = compute_ema(mkt; window = L_short);
        ema_l = compute_ema(mkt; window = L_long);
        λ_p = compute_lambda(ema_s, ema_l; G = GAIN);
        λ_p[1:offset] .= 0.0;
        gm_raw = compute_market_growth(mkt; Δt = Δt);
        gm_e = compute_ema(gm_raw; window = L_growth);

        pmatrix = zeros(scenario.n_steps, N + 1);
        pmatrix[:, 1] = 1:scenario.n_steps;
        for k in 1:N
            pmatrix[:, k + 1] = scenario.price_paths[p, :, k];
        end

        ctx = build(MyRebalancingContextModel, (
            B = B₀, tickers = my_tickers, marketdata = pmatrix,
            marketfactor = gm_e, sim_parameters = sim_params,
            lambda = 0.0, Δt = Δt, epsilon = ALLOCATION_EPSILON
        ));
        rules = build(MyTriggerRules, (
            max_drawdown = TRIGGER_MAX_DRAWDOWN, max_turnover = TRIGGER_MAX_TURNOVER,
            rebalance_schedule = ones(Int, n_trading)
        ));
        results = run_rebalancing_engine(ctx, rules, λ_p;
            offset = offset, allocator = :ces, adaptive_eta = true, eta_bounds = ETA_BOUNDS);
        w = compute_wealth_series(results, pmatrix, my_tickers; offset = offset);

        heur_final_wealth[p] = w[end];
        ret = diff(w) ./ w[1:end-1];
        pk = accumulate(max, w);
        heur_max_dd[p] = maximum((pk .- w) ./ pk);
        vol = std(ret) * sqrt(252);
        heur_sharpe[p] = vol > 0 ? (w[end]/w[1] - 1.0) / vol : 0.0;
    end

    # --- Step 3: Load frozen baseline from Example 1 ---
    ewls_data = load_results(joinpath(_PATH_TO_DATA, "ewls-replay-results.jld2"));
    frozen_fw = ewls_data["frozen_final_wealth"];

    # --- Step 4: Scorecard ---
    println("="^70)
    println("  CES ETA STRATEGIES: Distributional Comparison ($(scenario.n_paths) paths)")
    println("="^70)
    println("  Metric               Bandit-η      Heuristic-η    Fixed η=2.0")
    println("  " * "-"^55)
    println("  Median W/W₀          $(round(median(bandit_bt.final_wealth)/B₀, digits=3))           $(round(median(heur_final_wealth)/B₀, digits=3))           $(round(median(frozen_fw)/B₀, digits=3))")
    println("  Median Max DD        $(round(median(bandit_bt.max_drawdowns)*100, digits=1))%           $(round(median(heur_max_dd)*100, digits=1))%           --")
    println("  Median Sharpe        $(round(median(bandit_bt.sharpe_ratios), digits=3))           $(round(median(heur_sharpe), digits=3))           --")

    # --- Step 5: Save for Example 3 ---
    save_results(joinpath(_PATH_TO_DATA, "eta-bandit-results.jld2"), Dict(
        "bandit_final_wealth" => bandit_bt.final_wealth,
        "bandit_max_dd" => bandit_bt.max_drawdowns,
        "bandit_sharpe" => bandit_bt.sharpe_ratios,
        "heur_final_wealth" => heur_final_wealth,
        "heur_max_dd" => heur_max_dd,
        "heur_sharpe" => heur_sharpe,
        "best_eta_per_regime" => bandit_result.best_eta_per_regime,
    ));
    println("\nSaved to eta-bandit-results.jld2")
    bandit_bt
end

___
## Task 3: Bandit-Learned Elasticity vs. Heuristic Elasticity: Head-to-Head
In this task, we compute the per-path excess wealth (bandit minus heuristic) and visualize the distribution. This paired comparison controls for path-level variation and isolates the effect of the elasticity selection method.

> __What should we see?__
>
> If the bandit discovered genuinely better elasticity values, the excess distribution should be right-shifted (positive median). If the heuristic was already near-optimal, the distribution should be centered near zero.

In the code block below, we load the saved bandit and heuristic terminal-wealth arrays, compute the paired excess and win rate, and plot the histogram with a dashed zero line and a coral median marker.

In [ ]:
let
    # --- Step 1: Load bandit and heuristic results ---
    data = load_results(joinpath(_PATH_TO_DATA, "eta-bandit-results.jld2"));
    bandit_fw = data["bandit_final_wealth"];
    heur_fw   = data["heur_final_wealth"];

    # --- Step 2: Paired excess ---
    excess = bandit_fw .- heur_fw;
    win_rate = mean(excess .> 0);

    println("Paired Excess (Bandit - Heuristic):")
    println("  Win rate:      $(round(win_rate*100, digits=1))%")
    println("  Median excess: \$$(round(median(excess), digits=0))  ($(round(median(excess)/B₀*100, digits=2))% of W₀)")
    println("  Mean excess:   \$$(round(mean(excess), digits=0))  ($(round(mean(excess)/B₀*100, digits=2))% of W₀)")

    # --- Panel 1: Paired-excess histogram ---
    p = histogram(excess ./ B₀, bins = 60, alpha = 0.7, color = :steelblue,
        xlabel = "Per-Path Excess (W_bandit - W_heuristic) / W₀",
        ylabel = "Count", title = "Bandit-η vs Heuristic-η: Paired Excess",
        legend = false, size = (800, 400))
    vline!(p, [0.0], lw = 2, c = :black, linestyle = :dash)
    vline!(p, [median(excess) / B₀], lw = 2, c = :coral, label = "")
    display(p)

    # --- Step 3: Report learned eta map ---
    eta_map = data["best_eta_per_regime"];
    println("\nLearned eta map:")
    for regime in [:bearish, :neutral, :bullish]
        println("  $(regime): η = $(eta_map[regime])")
    end
end;

___
## Summary
This example trained an epsilon-greedy elasticity bandit on a single path, applied the learned elasticity map across 500 Monte Carlo paths, and compared the results head-to-head against the Session 2 heuristic. The learned elasticity map and backtest results are saved to `eta-bandit-results.jld2` for the validation report in Example 3.

> __Key Takeaways:__
>
> * __The bandit discovers regime-specific elasticity values:__ Different sentiment regimes may prefer different elasticity levels, and the bandit can break the symmetry that the heuristic imposes through its absolute-lambda dependence. Per-regime arm means make the preferred elasticity in each bin directly readable.
> * __Per-regime learning isolates the effect of elasticity choice:__ By holding preference weights and trigger rules fixed across the bandit, heuristic, and fixed-elasticity engines, the backtest comparison isolates how much the elasticity selection method contributes to performance. The remaining cross-strategy variance is attributable to elasticity alone.
> * __The heuristic is a strong baseline:__ If the excess distribution centers near zero, the hand-designed formula was already close to optimal on this data. The bandit's value then lies in confirming that conclusion empirically rather than in discovering a new operating point.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.

___